# 02 - Native Data Structures: Lists, Tuples, Dicts, Sets

Part of the Python, DSA & Git chapter. This notebook is arguably the single most interview-relevant piece of "pure Python" knowledge - nearly every DSA problem is really "which of these four structures fits this problem, and do I actually know its operation costs." Complexity tables below are worth memorizing cold.

## Lists: dynamic arrays

A Python `list` is a dynamic array (like `ArrayList` in Java or `vector` in C++), not a linked list - indexing is O(1), and it over-allocates capacity so `append` is amortized O(1).

In [1]:
nums = [5, 3, 8, 1, 9]

print(nums[0], nums[-1])           # O(1) indexing from either end
print(nums[1:3])                   # slicing -> new list [3, 8]
print(nums[::-1])                  # reversed copy

nums.append(2)                     # O(1) amortized
nums.insert(0, 100)                # O(n) -- has to shift every element right
print(nums)

nums.remove(100)                   # O(n) -- searches, then shifts
last = nums.pop()                  # O(1) -- pop from the END
first = nums.pop(0)                # O(n) -- pop from the FRONT shifts everything
print("removed:", last, first, "->", nums)

nums.sort()                        # O(n log n), in-place, Timsort
print(nums)
print(sorted(nums, reverse=True))  # sorted() returns a NEW list, leaves original untouched

# List comprehension -- the idiomatic, usually-fastest way to build a list
squares = [x**2 for x in range(10) if x % 2 == 0]
print(squares)

# Nested lists & the classic shallow-copy trap
matrix_wrong = [[0] * 3] * 3          # DANGER: same inner list repeated 3 times by reference
matrix_wrong[0][0] = 99
print("wrong (all rows changed!):", matrix_wrong)

matrix_right = [[0] * 3 for _ in range(3)]   # correct: fresh inner list each time
matrix_right[0][0] = 99
print("right (only row 0 changed):", matrix_right)

5 9
[3, 8]
[9, 1, 8, 3, 5]
[100, 5, 3, 8, 1, 9, 2]
removed: 2 5 -> [3, 8, 1, 9]
[1, 3, 8, 9]
[9, 8, 3, 1]
[0, 4, 16, 36, 64]
wrong (all rows changed!): [[99, 0, 0], [99, 0, 0], [99, 0, 0]]
right (only row 0 changed): [[99, 0, 0], [0, 0, 0], [0, 0, 0]]


### List operation complexity (memorize this table)

| Operation | Complexity | Note |
|---|---|---|
| `lst[i]` (index) | O(1) | direct memory offset |
| `lst.append(x)` | O(1) amortized | occasional O(n) resize, averages out |
| `lst.pop()` (from end) | O(1) | |
| `lst.pop(0)` / `lst.insert(0, x)` | O(n) | shifts every remaining element |
| `x in lst` | O(n) | linear scan |
| `lst.sort()` | O(n log n) | Timsort |
| `len(lst)` | O(1) | length is cached, not counted |
| slicing `lst[a:b]` | O(b-a) | copies the sliced range |

## Tuples: immutable, hashable sequences

Same indexing/slicing as lists, but immutable - which means, unlike lists, tuples can be dict keys or set members. Use a tuple when the data is a fixed-shape record (a coordinate, an RGB triple) rather than a growable collection.

In [2]:
point = (3, 4)
print(point[0], point[1])

try:
    point[0] = 99
except TypeError as e:
    print("tuples are immutable:", e)

# Tuple unpacking -- extremely common in idiomatic Python
x, y = point
print(x, y)

# Tuples ARE hashable (if their contents are) -> usable as dict keys / set members
visited = {(0, 0), (1, 1)}
print((0, 0) in visited)

# Named tuples: readable, still lightweight, still immutable
from collections import namedtuple
Point = namedtuple("Point", ["x", "y"])
p = Point(3, 4)
print(p.x, p.y, p)

# A dataclass with frozen=True is the modern equivalent when you also want methods
from dataclasses import dataclass

@dataclass(frozen=True)
class Point2:
    x: int
    y: int

    def distance_from_origin(self):
        return (self.x**2 + self.y**2) ** 0.5

p2 = Point2(3, 4)
print(p2, p2.distance_from_origin())

3 4
tuples are immutable: 'tuple' object does not support item assignment
3 4
True
3 4 Point(x=3, y=4)
Point2(x=3, y=4) 5.0


## Dictionaries: hash maps

Python's `dict` is a hash table - average O(1) lookup/insert/delete. Since Python 3.7, dicts preserve **insertion order** as a language guarantee (not just a CPython implementation detail anymore). Keys must be hashable, which is exactly why lists can't be dict keys but tuples can.

In [3]:
ages = {"alice": 30, "bob": 25}
ages["carol"] = 35                  # O(1) average insert
print(ages["alice"])                # O(1) average lookup
print(ages.get("dave", "unknown"))  # .get() with a default -- avoids KeyError

try:
    ages["dave"]
except KeyError as e:
    print("plain [] raises on missing key:", e)

print(list(ages.keys()))
print(list(ages.values()))
print(list(ages.items()))

for name, age in ages.items():      # the idiomatic way to iterate a dict
    print(name, age)

# Dict comprehension
squared = {x: x**2 for x in range(5)}
print(squared)

# defaultdict -- avoids manual "if key not in dict" boilerplate
from collections import defaultdict
groups = defaultdict(list)
for name, age in [("alice", 30), ("bob", 25), ("carol", 30)]:
    groups[age].append(name)        # no need to check/initialize first
print(dict(groups))

# Counter -- purpose-built for frequency counting, extremely common in DSA problems
from collections import Counter
word_counts = Counter("mississippi")
print(word_counts)
print(word_counts.most_common(2))   # top 2 most frequent characters

# setdefault -- get-or-insert in one call
d = {}
d.setdefault("key", []).append("value1")
d.setdefault("key", []).append("value2")
print(d)

30
unknown
plain [] raises on missing key: 'dave'
['alice', 'bob', 'carol']
[30, 25, 35]
[('alice', 30), ('bob', 25), ('carol', 35)]
alice 30
bob 25
carol 35
{0: 0, 1: 1, 2: 4, 3: 9, 4: 16}
{30: ['alice', 'carol'], 25: ['bob']}
Counter({'i': 4, 's': 4, 'p': 2, 'm': 1})
[('i', 4), ('s', 4)]
{'key': ['value1', 'value2']}


### Dict operation complexity

| Operation | Average | Worst case |
|---|---|---|
| `d[key]` get/set | O(1) | O(n) (hash collisions -- rare in practice) |
| `key in d` | O(1) | O(n) |
| `del d[key]` | O(1) | O(n) |
| iterating all items | O(n) | O(n) |

**Why average O(1) and not always:** a dict hashes the key to find a bucket directly, instead of scanning. Worst case only shows up with pathological hash collisions, which Python's hash randomization makes very unlikely to hit by accident.

## Sets: hash tables with no values, only keys

A `set` is essentially a dict with only keys - same O(1) average membership testing, no ordering guarantee, no duplicates. The single most common DSA use: checking "have I seen this before?" in O(1) instead of O(n).

In [4]:
a = {1, 2, 3, 4}
b = {3, 4, 5, 6}

print(a | b)   # union
print(a & b)   # intersection
print(a - b)   # difference (in a, not in b)
print(a ^ b)   # symmetric difference (in exactly one of them)

print(3 in a)  # O(1) average membership test -- THE reason sets show up constantly in DSA

# Classic pattern: dedup while preserving nothing about order (sets are unordered)
nums = [1, 2, 2, 3, 3, 3, 4]
print(set(nums))

# frozenset -- the immutable, hashable version (usable as a dict key or set member)
fs = frozenset([1, 2, 3])
cache = {fs: "some cached result"}
print(cache[frozenset([1, 2, 3])])

# Why "x in list" vs "x in set" matters at scale
import timeit
big_list = list(range(100_000))
big_set = set(big_list)

t_list = timeit.timeit(lambda: 99_999 in big_list, number=100)
t_set = timeit.timeit(lambda: 99_999 in big_set, number=100)
print(f"'in' on a 100k-element list: {t_list:.5f}s")
print(f"'in' on a 100k-element set:  {t_set:.5f}s  (should be dramatically faster)")

{1, 2, 3, 4, 5, 6}
{3, 4}
{1, 2}
{1, 2, 5, 6}
True
{1, 2, 3, 4}
some cached result
'in' on a 100k-element list: 0.11010s
'in' on a 100k-element set:  0.00001s  (should be dramatically faster)


### Set operation complexity

| Operation | Average |
|---|---|
| `x in s` | O(1) |
| `s.add(x)` / `s.remove(x)` | O(1) |
| union / intersection / difference | O(len(smaller set)) roughly |

**The single highest-leverage DSA habit:** whenever a problem says "check if X has been seen before" or "find duplicates" or "check membership repeatedly," your first instinct should be *"can I use a set (or dict) to make this O(1) instead of O(n) per check?"* This one habit turns a huge fraction of brute-force O(n²) solutions into O(n).

## Practice exercises

Fill in each TODO. These specifically exercise the "reach for the right structure" instinct.

In [5]:
def has_duplicate(nums):
    # Return True if any value appears more than once in nums.
    # TODO: implement in O(n) using a set
    raise NotImplementedError

def first_non_repeating_char(s):
    # Return the first character in s that appears exactly once. Return None if none exists.
    # TODO: implement using a dict/Counter for O(n)
    raise NotImplementedError

def group_anagrams(words):
    # Group words that are anagrams of each other. Return a list of lists (any order).
    # e.g. ["eat","tea","tan","ate","nat","bat"] -> [["eat","tea","ate"],["tan","nat"],["bat"]]
    # TODO: implement using a dict keyed by sorted-letters
    raise NotImplementedError

def two_sum(nums, target):
    # Return the (i, j) indices of two numbers that add up to target (i < j), or None.
    # TODO: implement in O(n) using a dict of value -> index (NOT the O(n^2) brute force)
    raise NotImplementedError

In [6]:
def _check(name, fn, cases, unpack=False):
    for args, expected in cases:
        try:
            result = fn(*args) if unpack else fn(args)
        except NotImplementedError:
            print("  [SKIP]", name, "-- not implemented yet")
            return
        except Exception as e:
            print("  [ERROR]", name, args, "--", e)
            return
        # for group_anagrams, compare as a set of frozensets so order doesn't matter
        if name == "group_anagrams":
            norm = lambda groups: set(frozenset(g) for g in groups)
            ok = norm(result) == norm(expected)
        else:
            ok = result == expected
        tag = "PASS" if ok else "FAIL"
        extra = "" if ok else ("expected " + repr(expected))
        print("  [" + tag + "]", name + "(" + repr(args) + ") ->", repr(result), extra)

_check("has_duplicate", has_duplicate, [([1, 2, 3, 1], True), ([1, 2, 3], False)])
_check("first_non_repeating_char", first_non_repeating_char,
       [("swiss", "w"), ("aabbcc", None)])
_check("group_anagrams", group_anagrams,
       [(["eat", "tea", "tan", "ate", "nat", "bat"],
         [["eat", "tea", "ate"], ["tan", "nat"], ["bat"]])])
_check("two_sum", two_sum, [(([2, 7, 11, 15], 9), (0, 1))], unpack=True)

  [SKIP] has_duplicate -- not implemented yet
  [SKIP] first_non_repeating_char -- not implemented yet
  [SKIP] group_anagrams -- not implemented yet
  [SKIP] two_sum -- not implemented yet


## Self-check before moving on

- [ ] I know the complexity of append/insert/pop for lists, and why front operations are O(n)
- [ ] I know dict/set operations are O(1) average, and I default to them for "have I seen this?" checks
- [ ] I can explain the `[[0]*3]*3` shallow-copy trap and how to avoid it
- [ ] I know when to reach for `Counter`, `defaultdict`, and `namedtuple` instead of hand-rolling the equivalent
- [ ] I can explain why tuples are hashable and lists aren't

Next: `03-functions-oop.ipynb`